# RI-JK RHF Hessian：简单项分解

In [1]:
from pyscf import gto, scf, lib
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = scf.RHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_r_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_r_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_r_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = np.dot(mocc, mocc.T) * 2
dme0 = np.einsum('pi,qi,i->pq', mocc, mocc, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)

In [6]:
de_nuc = np.load("nh3_r_hf_decomp.npz")["de_nuc"]
de_1 = np.load("nh3_r_hf_decomp.npz")["de_1"]

## 原子核贡献计算

In [7]:
h = np.zeros((natm, natm, 3, 3))
qs = np.asarray([mol.atom_charge(i) for i in range(natm)])
rs = np.asarray([mol.atom_coord(i) for i in range(natm)])
for i in range(natm):
    r12 = rs[i] - rs                          # shape (natm, 3)
    s12 = np.sqrt(np.sum(r12 * r12, axis=1))  # einsum: 'ki,ki->k'
    s12[i] = np.inf
    tmp1 = qs[i] * qs / s12**3                # shape (natm,)
    prefactor = -3 * qs[i] * qs / s12**5      # shape (natm,)
    tmp2 = (prefactor[:, None, None] * r12[:, :, None] * r12[:, None, :])  # einsum: 'k,ki,kj->kij'

    # Diagonal block h[i,i]
    h[i, i, 0, 0] = h[i, i, 1, 1] = h[i, i, 2, 2] = -tmp1.sum()
    h[i, i] -= np.sum(tmp2, axis=0)           # einsum: 'kij->ij'

    # Off-diagonal blocks h[i,:] for all k
    h[i, :, 0, 0] += tmp1
    h[i, :, 1, 1] += tmp1
    h[i, :, 2, 2] += tmp1
    h[i, :] += tmp2

In [8]:
np.allclose(h, de_nuc)

True

## hcore 分解

In [9]:
def hcore_deriv2_generator(mol):
    # preparation
    nao = mol.nao
    nbas = mol.nbas
    aoslices = mol.aoslice_by_atom()
    ecp_atoms = set(mol._ecpbas[:, gto.ATOM_OF])
    # we need to prepare some integrals, to somehow avoid redundant calculations in the loop
    # - aa: Hamiltonian derivative to only the first basis
    # - ab: Hamiltonian derivative to the first and second basis
    
    h2_aa = mol.intor("int1e_ipipkin").reshape(3, 3, nao, nao)
    h2_ab = mol.intor("int1e_ipkinip").reshape(3, 3, nao, nao)
    h2_aa += mol.intor("int1e_ipipnuc").reshape(3, 3, nao, nao)
    h2_ab += mol.intor("int1e_ipnucip").reshape(3, 3, nao, nao)
    if mol.has_ecp():
        h2_aa += mol.intor("ECPscalar_ipipnuc").reshape(3, 3, nao, nao)
        h2_ab += mol.intor("ECPscalar_ipnucip").reshape(3, 3, nao, nao)
    def get_hcore_deriv_at_atoms(A, B):
        sh0A, sh1A, p0A, p1A = aoslices[A]
        sh0B, sh1B, p0B, p1B = aoslices[B]
        slcA = slice(p0A, p1A)
        slcB = slice(p0B, p1B)
        zi = mol.atom_charge(A)
        hcore_deriv = np.zeros((3, 3, nao, nao))

        if A == B:
            with mol.with_rinv_at_nucleus(A):
                rinv_aa = -zi * mol.intor("int1e_ipiprinv").reshape(3, 3, nao, nao)
                rinv_ab = -zi * mol.intor("int1e_iprinvip").reshape(3, 3, nao, nao)
                if A in ecp_atoms:
                    rinv_aa += mol.intor("ECPscalar_ipiprinv").reshape(3, 3, nao, nao)
                    rinv_ab += mol.intor("ECPscalar_iprinvip").reshape(3, 3, nao, nao)
                hcore_deriv += (rinv_aa + rinv_ab)
                hcore_deriv[:, :, slcA, :] += h2_aa[:, :, slcA, :]
                hcore_deriv[:, :, slcA, slcA] += h2_ab[:, :, slcA, slcA]
                hcore_deriv[:, :, slcA, :] -= rinv_aa[:, :, slcA, :]
                hcore_deriv[:, :, slcA, :] -= rinv_ab[:, :, slcA, :]
                hcore_deriv[:, :, :, slcA] -= rinv_aa[:, :, slcA, :].swapaxes(-1, -2)
                hcore_deriv[:, :, :, slcA] -= rinv_ab[:, :, :, slcA]
        else:
            hcore_deriv[:, :, slcA, slcB] += h2_ab[:, :, slcA, slcB]
            # handle rinv@i, basis@j
            zi = mol.atom_charge(A)
            with mol.with_rinv_at_nucleus(A):
                shls_slice = (sh0B, sh1B, 0, nbas)
                rinv_atom_aa = -zi * mol.intor("int1e_ipiprinv", shls_slice=shls_slice).reshape(3, 3, -1, nao)
                rinv_atom_ab = -zi * mol.intor("int1e_iprinvip", shls_slice=shls_slice).reshape(3, 3, -1, nao)
                if A in ecp_atoms:
                    rinv_atom_aa += mol.intor("ECPscalar_ipiprinv", shls_slice=shls_slice).reshape(3, 3, -1, nao)
                    rinv_atom_ab += mol.intor("ECPscalar_iprinvip", shls_slice=shls_slice).reshape(3, 3, -1, nao)
            hcore_deriv[:, :, slcB, :] -= rinv_atom_aa
            hcore_deriv[:, :, slcB, :] -= rinv_atom_ab.swapaxes(0, 1)
            # handle rinv@j, basis@i
            zj = mol.atom_charge(B)
            with mol.with_rinv_at_nucleus(B):
                shls_slice = (sh0A, sh1A, 0, nbas)
                rinv_atom_aa = -zj * mol.intor("int1e_ipiprinv", shls_slice=shls_slice).reshape(3, 3, -1, nao)
                rinv_atom_ab = -zj * mol.intor("int1e_iprinvip", shls_slice=shls_slice).reshape(3, 3, -1, nao)
                if B in ecp_atoms:
                    rinv_atom_aa += mol.intor("ECPscalar_ipiprinv", shls_slice=shls_slice).reshape(3, 3, -1, nao)
                    rinv_atom_ab += mol.intor("ECPscalar_iprinvip", shls_slice=shls_slice).reshape(3, 3, -1, nao)
            hcore_deriv[:, :, slcA, :] -= rinv_atom_aa
            hcore_deriv[:, :, slcA, :] -= rinv_atom_ab
            
        hcore_deriv += hcore_deriv.swapaxes(-1, -2)
        return hcore_deriv
    return get_hcore_deriv_at_atoms

In [10]:
for i in range(4):
    for j in range(4):
        assert np.allclose(hcore_deriv2_generator(mol)(i, j), mf_hess.hcore_generator()(i, j))

## overlap 分解与一阶项导数

In [11]:
hcore_deriv2_closure = hcore_deriv2_generator(mol)
s1_aa = mol.intor("int1e_ipipovlp").reshape(3, 3, nao, nao)
s1_ab = mol.intor("int1e_ipovlpip").reshape(3, 3, nao, nao)

de_hcore = np.zeros((natm, natm, 3, 3))
de_ovlp = np.zeros((natm, natm, 3, 3))

for A in range(natm):
    sh0A, sh1A, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    # de_ovlp[A, A] -= 2 * np.einsum('tsuv, uv -> ts', s1_aa[:, :, slcA], dme0[slcA])
    de_ovlp[A, A] -= 2 * (s1_aa[:, :, slcA, :] * dme0[slcA, :]).sum(axis=(-1, -2))
    for B in range(A + 1):
        _, _, p0B, p1B = aoslices[B]
        slcB = slice(p0B, p1B)
        hcore_deriv2 = hcore_deriv2_closure(A, B)
        # de_hcore[A, B] += np.einsum('tsuv, uv -> ts', hcore_deriv2, dm0)
        de_hcore[A, B] += (hcore_deriv2 * dm0).sum(axis=(-1, -2))
        # de_ovlp[A, B] -= 2 * np.einsum('tsuv, uv -> ts', s1_ab[:, :, slcA, slcB], dme0[slcA, slcB])
        de_ovlp[A, B] -= 2 * (s1_ab[:, :, slcA, slcB] * dme0[slcA, slcB]).sum(axis=(-1, -2))
    for B in range(A):
        de_hcore[B, A] = de_hcore[A, B].T
        de_ovlp[B, A] = de_ovlp[A, B].T

np.allclose(de_hcore + de_ovlp, de_1)

True

## 存储到分解文件

In [12]:
dat = dict(np.load("nh3_r_hf_decomp.npz"))
dat.update({
    "de_hcore": de_hcore,
    "de_ovlp": de_ovlp,
})
np.savez("nh3_r_hf_decomp.npz", **dat)